In [15]:
from langchain_ollama import ChatOllama
from typing import TypedDict, Literal
from langgraph.graph import StateGraph,START,END
from pydantic import BaseModel,Field



In [16]:
llm = ChatOllama(model='qwen2.5:1.5b',temperature=0.2,max_token=200)

In [30]:
class QuerytypeSchema(BaseModel):
    type:Literal['factual','opinion','how-to'] = Field(description="type of user query")

class StateSchema(TypedDict):
    query: str
    type: str
    answer: str
  
    

In [31]:
def classify_query(state:StateSchema):
    structured_response_llm = llm.with_structured_output(QuerytypeSchema)
    prompt = f"""find the type of the query - {state['query']}."""
    res = structured_response_llm.invoke(prompt)
    return {'type': res.type}

def explain_node(state:StateSchema):
    prompt = f"""generate an small explaination for the query - {state['query']}."""
    res = llm.invoke(prompt)
    # print("====== explain node ====",res)
    return {'answer': res.content}

def opinion_node(state:StateSchema):
    prompt = f"""generate an small opinion-style response for the query - {state['query']}."""
    res = llm.invoke(prompt)
    # print("====== opinion node ====",res)
    return {'answer': res.content}

def guide_node(state:StateSchema):
    prompt = f"""generate an small step by step guaidance for the query - {state['query']}."""
    res = llm.invoke(prompt)
    # print("====== guide node ====",res)
    return {'answer': res.content}

def check_condition(state:StateSchema):
    if state['type'] == 'factual':
        return 'explain_node'
    elif state['type'] == 'opinion':
        return 'opinion_node'
    return 'guide_node'

    


In [33]:
graph = StateGraph(StateSchema)
graph.add_node('classify_query_node',classify_query)
graph.add_node('explain_node',explain_node)
graph.add_node('opinion_node',opinion_node)
graph.add_node('guide_node',guide_node)

graph.add_edge(START,'classify_query_node')
graph.add_conditional_edges('classify_query_node',check_condition,{'explain_node':'explain_node','opinion_node':'opinion_node','guide_node':'guide_node'})
graph.add_edge('explain_node',END)
graph.add_edge('opinion_node',END)
graph.add_edge('guide_node',END)
workflow = graph.compile()

In [34]:
print(workflow.get_graph().draw_ascii())

                            +-----------+                            
                            | __start__ |                            
                            +-----------+                            
                                  *                                  
                                  *                                  
                                  *                                  
                      +---------------------+                        
                      | classify_query_node |                        
                      +---------------------+                        
                   .....          .          .....                   
               ....               .               ....               
            ...                   .                   ...            
+--------------+           +------------+           +--------------+ 
| explain_node |           | guide_node |           | opinion_node | 
+--------------+*** 

In [35]:
res = workflow.invoke({'query':'How to buid a langgraph'})
res

{'query': 'How to buid a langgraph',
 'type': 'how-to',
 'answer': 'Building a language graph involves several steps that can be broken down into smaller tasks. Here\'s a simple guide:\n\n### Step 1: Define Your Topic or Domain\n- **Identify your focus**: Decide what topic you want to build a language graph around, such as "technology trends," "healthcare innovations," or any other specific area.\n- **Research the domain**: Understand key concepts and terminology within this field.\n\n### Step 2: Collect Data\n- **Gather information**: Look for existing resources like articles, reports, news articles, academic papers, and industry publications. These can provide a wealth of data points.\n- **Analyze trends**: Identify patterns or changes over time in your chosen topic. This could involve tracking mentions of specific terms or technologies.\n\n### Step 3: Create Nodes\n- **Identify key concepts**: For each piece of information you\'ve gathered, identify the main concept it represents (e